# EDA e integración de datos: Spain 2024/2025

Objetivo:
- Analizar los datos de rendimiento de Sofascore y salarios de Capology para La Liga 2024/2025.
- Limpiar y normalizar ambas fuentes.
- Unificar los datos en un único dataset.
- Realizar un análisis exploratorio inicial.
- Dejar preparado un dataset base para futuros modelos de machine learning.

In [81]:
import pandas as pd
import numpy as np
import re
import unicodedata
import matplotlib.pyplot as plt

In [82]:
df_perf_raw = pd.read_csv("data/raw/sofascore/df_spain_2425.csv")
df_sal_raw = pd.read_csv("data/raw/capology/cg_spain_2425.csv")

In [83]:
print("=== SOFASCORE ===")
print(df_perf_raw.shape)
display(df_perf_raw.head())
display(df_perf_raw.info())

print("\n=== CAPOLOGY ===")
print(df_sal_raw.shape)
display(df_sal_raw.head(10))
display(df_sal_raw.info())

=== SOFASCORE ===
(589, 116)


,accurateChippedPasses,accurateCrosses,accurateCrossesPercentage,accurateFinalThirdPasses,accurateLongBalls,accurateLongBallsPercentage,accurateOppositionHalfPasses,accurateOwnHalfPasses,accuratePasses,accuratePassesPercentage,...,touches,wasFouled,yellowCards,yellowRedCards,player,team,player id,team id,data_country,data_season
0,3,0,0.00,3,19,54.29,13,12,25,60.98,...,53,0,0,0,Aitor Fernández,Osasuna,99516,2820,spain,2425
1,8,0,0.00,11,63,38.41,40,79,119,53.60,...,299,1,0,0,Leo Román,Mallorca,1131909,2826,spain,2425
2,62,53,24.65,511,46,58.97,838,209,994,79.46,...,1993,37,4,0,Raphinha,Barcelona,831005,2817,spain,2425
3,45,22,20.37,633,30,48.39,980,158,1116,78.93,...,2361,60,3,0,Lamine Yamal,Barcelona,1402912,2817,spain,2425
4,49,6,17.14,573,45,78.95,800,139,933,85.28,...,1734,40,3,0,Kylian Mbappé,Real Madrid,826643,2829,spain,2425


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 589 entries, 0 to 588
Columns: 116 entries, accurateChippedPasses to data_season
dtypes: float64(18), int64(95), object(3)
memory usage: 533.9+ KB


None


=== CAPOLOGY ===
(552, 10)


,Unnamed: 0,EST. BASE SALARY,EST. BASE SALARY.1,EST. BASE SALARY.2,BIO,BIO.1,BIO.2,Unnamed: 7,data_country,data_season
0,PLAYER,GROSS P/W\n(EUR),GROSS P/Y\n(EUR),ADJ. GROSS\n(EUR),POS.,AGE,COUNTRY,CLUB,NaN,NaN
1,Robert Lewandowski,"€ 640,962","€ 33,330,000","€ 33,330,000",F,36,Poland,Barcelona,spain,2425.0
2,Kylian Mbappé,"€ 600,962","€ 31,250,000","€ 31,250,000",F,26,France,Real Madrid,spain,2425.0
3,Vinicius Junior,"€ 480,769","€ 25,000,000","€ 25,000,000",F,24,Brazil,Real Madrid,spain,2425.0
4,David Alaba,"€ 432,692","€ 22,500,000","€ 22,500,000",D,32,Austria,Real Madrid,spain,2425.0
5,Jan Oblak,"€ 400,577","€ 20,830,000","€ 20,830,000",K,32,Slovenia,Atletico Madrid,spain,2425.0
6,Jude Bellingham,"€ 400,577","€ 20,830,000","€ 20,830,000",F,21,England,Real Madrid,spain,2425.0
7,Frenkie de Jong,"€ 365,385","€ 19,000,000","€ 19,000,000",M,27,Netherlands,Barcelona,spain,2425.0
8,Federico Valverde,"€ 320,577","€ 16,670,000","€ 16,670,000",M,26,Uruguay,Real Madrid,spain,2425.0
9,Rodrygo,"€ 320,577","€ 16,670,000","€ 16,670,000",F,24,Brazil,Real Madrid,spain,2425.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 552 entries, 0 to 551
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Unnamed: 0          552 non-null    object 
 1   EST. BASE SALARY    552 non-null    object 
 2   EST. BASE SALARY.1  552 non-null    object 
 3   EST. BASE SALARY.2  552 non-null    object 
 4   BIO                 552 non-null    object 
 5   BIO.1               552 non-null    object 
 6   BIO.2               552 non-null    object 
 7   Unnamed: 7          552 non-null    object 
 8   data_country        551 non-null    object 
 9   data_season         551 non-null    float64
dtypes: float64(1), object(9)
memory usage: 43.3+ KB


None

In [84]:
df_perf = df_perf_raw.copy()
df_sal = df_sal_raw.copy()

In [85]:
df_sal.head(5)

,Unnamed: 0,EST. BASE SALARY,EST. BASE SALARY.1,EST. BASE SALARY.2,BIO,BIO.1,BIO.2,Unnamed: 7,data_country,data_season
0,PLAYER,GROSS P/W\n(EUR),GROSS P/Y\n(EUR),ADJ. GROSS\n(EUR),POS.,AGE,COUNTRY,CLUB,NaN,NaN
1,Robert Lewandowski,"€ 640,962","€ 33,330,000","€ 33,330,000",F,36,Poland,Barcelona,spain,2425.0
2,Kylian Mbappé,"€ 600,962","€ 31,250,000","€ 31,250,000",F,26,France,Real Madrid,spain,2425.0
3,Vinicius Junior,"€ 480,769","€ 25,000,000","€ 25,000,000",F,24,Brazil,Real Madrid,spain,2425.0
4,David Alaba,"€ 432,692","€ 22,500,000","€ 22,500,000",D,32,Austria,Real Madrid,spain,2425.0


In [86]:
# Usar la primera fila como cabecera
df_sal.columns = df_sal.iloc[0]
df_sal = df_sal.iloc[1:].reset_index(drop=True)

# Limpiar nombres de columnas
df_sal.columns = [str(col).strip() for col in df_sal.columns]

display(df_sal.head())
print(df_sal.shape)
print(df_sal.columns.tolist())

,PLAYER,GROSS P/W\n(EUR),GROSS P/Y\n(EUR),ADJ. GROSS\n(EUR),POS.,AGE,COUNTRY,CLUB,nan,nan
0,Robert Lewandowski,"€ 640,962","€ 33,330,000","€ 33,330,000",F,36,Poland,Barcelona,spain,2425.0
1,Kylian Mbappé,"€ 600,962","€ 31,250,000","€ 31,250,000",F,26,France,Real Madrid,spain,2425.0
2,Vinicius Junior,"€ 480,769","€ 25,000,000","€ 25,000,000",F,24,Brazil,Real Madrid,spain,2425.0
3,David Alaba,"€ 432,692","€ 22,500,000","€ 22,500,000",D,32,Austria,Real Madrid,spain,2425.0
4,Jan Oblak,"€ 400,577","€ 20,830,000","€ 20,830,000",K,32,Slovenia,Atletico Madrid,spain,2425.0


(551, 10)
['PLAYER', 'GROSS P/W\n(EUR)', 'GROSS P/Y\n(EUR)', 'ADJ. GROSS\n(EUR)', 'POS.', 'AGE', 'COUNTRY', 'CLUB', 'nan', 'nan']


In [87]:
print(df_sal.columns.tolist())

['PLAYER', 'GROSS P/W\n(EUR)', 'GROSS P/Y\n(EUR)', 'ADJ. GROSS\n(EUR)', 'POS.', 'AGE', 'COUNTRY', 'CLUB', 'nan', 'nan']


In [88]:
cols_sal_keep = [
    "PLAYER",
    "GROSS P/W\n(EUR)",
    "GROSS P/Y\n(EUR)",
    "ADJ. GROSS\n(EUR)",
    "POS.",
    "AGE",
    "COUNTRY",
    "CLUB"
]

df_sal = df_sal[cols_sal_keep].copy()
display(df_sal.head())

,PLAYER,GROSS P/W\n(EUR),GROSS P/Y\n(EUR),ADJ. GROSS\n(EUR),POS.,AGE,COUNTRY,CLUB
0,Robert Lewandowski,"€ 640,962","€ 33,330,000","€ 33,330,000",F,36,Poland,Barcelona
1,Kylian Mbappé,"€ 600,962","€ 31,250,000","€ 31,250,000",F,26,France,Real Madrid
2,Vinicius Junior,"€ 480,769","€ 25,000,000","€ 25,000,000",F,24,Brazil,Real Madrid
3,David Alaba,"€ 432,692","€ 22,500,000","€ 22,500,000",D,32,Austria,Real Madrid
4,Jan Oblak,"€ 400,577","€ 20,830,000","€ 20,830,000",K,32,Slovenia,Atletico Madrid


In [89]:
rename_sal = {
    "PLAYER": "player",
    "GROSS P/W\n(EUR)": "weekly_salary",
    "GROSS P/Y\n(EUR)": "yearly_salary",
    "ADJ. GROSS\n(EUR)": "adjusted_gross_salary",
    "POS.": "position_capology",
    "AGE": "age",
    "COUNTRY": "country",
    "CLUB": "team"
}

df_sal = df_sal.rename(columns=rename_sal)
print(df_sal.columns.tolist())

['player', 'weekly_salary', 'yearly_salary', 'adjusted_gross_salary', 'position_capology', 'age', 'country', 'team']


In [90]:
def clean_currency(x):
    if pd.isna(x):
        return np.nan
    x = str(x)
    x = x.replace("€", "").replace(",", "").strip()
    if x == "" or x == "-":
        return np.nan
    try:
        return float(x)
    except:
        return np.nan

df_sal["weekly_salary"] = df_sal["weekly_salary"].apply(clean_currency)
df_sal["yearly_salary"] = df_sal["yearly_salary"].apply(clean_currency)
df_sal["age"] = pd.to_numeric(df_sal["age"], errors="coerce")

In [91]:
df_sal[["player", "team", "weekly_salary", "yearly_salary", "age"]].head()

,player,team,weekly_salary,yearly_salary,age
0,Robert Lewandowski,Barcelona,640962.0,33330000.0,36
1,Kylian Mbappé,Real Madrid,600962.0,31250000.0,26
2,Vinicius Junior,Real Madrid,480769.0,25000000.0,24
3,David Alaba,Real Madrid,432692.0,22500000.0,32
4,Jan Oblak,Atletico Madrid,400577.0,20830000.0,32


In [92]:
print(df_sal.shape)
print(df_sal.isna().sum().sort_values(ascending=False).head(10))
print("Duplicados por jugador-equipo:", df_sal.duplicated(subset=["player", "team"]).sum())

(551, 8)
player                   0
weekly_salary            0
yearly_salary            0
adjusted_gross_salary    0
position_capology        0
age                      0
country                  0
team                     0
dtype: int64
Duplicados por jugador-equipo: 0


In [93]:
print(df_perf.shape)
print(df_perf.columns.tolist())
display(df_perf.head())

(589, 116)
['accurateChippedPasses', 'accurateCrosses', 'accurateCrossesPercentage', 'accurateFinalThirdPasses', 'accurateLongBalls', 'accurateLongBallsPercentage', 'accurateOppositionHalfPasses', 'accurateOwnHalfPasses', 'accuratePasses', 'accuratePassesPercentage', 'aerialDuelsWon', 'aerialDuelsWonPercentage', 'aerialLost', 'appearances', 'assists', 'attemptPenaltyMiss', 'attemptPenaltyPost', 'attemptPenaltyTarget', 'ballRecovery', 'bigChancesCreated', 'bigChancesMissed', 'blockedShots', 'cleanSheet', 'clearances', 'countRating', 'crossesNotClaimed', 'directRedCards', 'dispossessed', 'dribbledPast', 'duelLost', 'errorLeadToGoal', 'errorLeadToShot', 'expectedAssists', 'expectedGoals', 'fouls', 'freeKickGoal', 'goalConversionPercentage', 'goalKicks', 'goals', 'goalsAssistsSum', 'goalsConceded', 'goalsConcededInsideTheBox', 'goalsConcededOutsideTheBox', 'goalsFromInsideTheBox', 'goalsFromOutsideTheBox', 'goalsPrevented', 'groundDuelsWon', 'groundDuelsWonPercentage', 'headedGoals', 'high

,accurateChippedPasses,accurateCrosses,accurateCrossesPercentage,accurateFinalThirdPasses,accurateLongBalls,accurateLongBallsPercentage,accurateOppositionHalfPasses,accurateOwnHalfPasses,accuratePasses,accuratePassesPercentage,...,touches,wasFouled,yellowCards,yellowRedCards,player,team,player id,team id,data_country,data_season
0,3,0,0.00,3,19,54.29,13,12,25,60.98,...,53,0,0,0,Aitor Fernández,Osasuna,99516,2820,spain,2425
1,8,0,0.00,11,63,38.41,40,79,119,53.60,...,299,1,0,0,Leo Román,Mallorca,1131909,2826,spain,2425
2,62,53,24.65,511,46,58.97,838,209,994,79.46,...,1993,37,4,0,Raphinha,Barcelona,831005,2817,spain,2425
3,45,22,20.37,633,30,48.39,980,158,1116,78.93,...,2361,60,3,0,Lamine Yamal,Barcelona,1402912,2817,spain,2425
4,49,6,17.14,573,45,78.95,800,139,933,85.28,...,1734,40,3,0,Kylian Mbappé,Real Madrid,826643,2829,spain,2425


In [94]:
priority_cols = [
    "player",
    "player id",
    "team id",
    "position",
    "age",
    "country",
    "team",
    "data_country",
    "data_season"
]

priority_cols = [col for col in priority_cols if col in df_perf.columns]
other_cols = [col for col in df_perf.columns if col not in priority_cols]

df_perf = df_perf[priority_cols + other_cols].copy()

display(df_perf.head())
print(df_perf.columns.tolist()[:20])  # para comprobar el nuevo orden

,player,player id,team id,team,data_country,data_season,accurateChippedPasses,accurateCrosses,accurateCrossesPercentage,accurateFinalThirdPasses,...,totalOppositionHalfPasses,totalOwnHalfPasses,totalPasses,totalRating,totalShots,totwAppearances,touches,wasFouled,yellowCards,yellowRedCards
0,Aitor Fernández,99516,2820,Osasuna,spain,2425,3,0,0.00,3,...,27,14,41,8.4,0,1,53,0,0,0
1,Leo Román,1131909,2826,Mallorca,spain,2425,8,0,0.00,11,...,126,96,222,54.8,0,3,299,1,0,0
2,Raphinha,831005,2817,Barcelona,spain,2425,62,53,24.65,511,...,1224,242,1251,280.7,114,14,1993,37,4,0
3,Lamine Yamal,1402912,2817,Barcelona,spain,2425,45,22,20.37,633,...,1330,192,1414,272.8,144,11,2361,60,3,0
4,Kylian Mbappé,826643,2829,Real Madrid,spain,2425,49,6,17.14,573,...,976,153,1094,261.7,161,12,1734,40,3,0


['player', 'player id', 'team id', 'team', 'data_country', 'data_season', 'accurateChippedPasses', 'accurateCrosses', 'accurateCrossesPercentage', 'accurateFinalThirdPasses', 'accurateLongBalls', 'accurateLongBallsPercentage', 'accurateOppositionHalfPasses', 'accurateOwnHalfPasses', 'accuratePasses', 'accuratePassesPercentage', 'aerialDuelsWon', 'aerialDuelsWonPercentage', 'aerialLost', 'appearances']


In [95]:
df_perf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 589 entries, 0 to 588
Columns: 116 entries, player to yellowRedCards
dtypes: float64(18), int64(95), object(3)
memory usage: 533.9+ KB


In [96]:
text_like_cols = {
    "player",
    "team",
    "position_sofascore",
    "country",
    "data_country"
}

for col in df_perf.columns:
    if col not in text_like_cols:
        df_perf[col] = pd.to_numeric(df_perf[col], errors="ignore")

df_perf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 589 entries, 0 to 588
Columns: 116 entries, player to yellowRedCards
dtypes: float64(18), int64(95), object(3)
memory usage: 533.9+ KB


C:\Users\User\AppData\Local\Temp\ipykernel_22864\960923657.py:11: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df_perf[col] = pd.to_numeric(df_perf[col], errors="ignore")


In [97]:
def normalize_text(x):
    if pd.isna(x):
        return np.nan
    
    x = str(x).strip().lower()
    
    replacements = {
        "ø": "o",
        "ß": "ss",
        "æ": "ae",
        "œ": "oe",
        "đ": "d",
        "ł": "l"
    }
    
    for old, new in replacements.items():
        x = x.replace(old, new)
    
    x = unicodedata.normalize("NFKD", x)
    x = "".join(c for c in x if not unicodedata.combining(c))
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    
    return x

In [98]:
df_sal["player_norm"] = df_sal["player"].apply(normalize_text)
df_sal["team_norm"] = df_sal["team"].apply(normalize_text)

df_perf["player_norm"] = df_perf["player"].apply(normalize_text)
df_perf["team_norm"] = df_perf["team"].apply(normalize_text)

C:\Users\User\AppData\Local\Temp\ipykernel_22864\4178972775.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_perf["player_norm"] = df_perf["player"].apply(normalize_text)
C:\Users\User\AppData\Local\Temp\ipykernel_22864\4178972775.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_perf["team_norm"] = df_perf["team"].apply(normalize_text)


In [99]:
display(df_sal[["player", "player_norm", "team", "team_norm"]].head(10))
display(df_perf[["player", "player_norm", "team", "team_norm"]].head(10))

,player,player_norm,team,team_norm
0,Robert Lewandowski,robert lewandowski,Barcelona,barcelona
1,Kylian Mbappé,kylian mbappe,Real Madrid,real madrid
2,Vinicius Junior,vinicius junior,Real Madrid,real madrid
3,David Alaba,david alaba,Real Madrid,real madrid
4,Jan Oblak,jan oblak,Atletico Madrid,atletico madrid
5,Jude Bellingham,jude bellingham,Real Madrid,real madrid
6,Frenkie de Jong,frenkie de jong,Barcelona,barcelona
7,Federico Valverde,federico valverde,Real Madrid,real madrid
8,Rodrygo,rodrygo,Real Madrid,real madrid
9,Raphinha,raphinha,Barcelona,barcelona


,player,player_norm,team,team_norm
0,Aitor Fernández,aitor fernandez,Osasuna,osasuna
1,Leo Román,leo roman,Mallorca,mallorca
2,Raphinha,raphinha,Barcelona,barcelona
3,Lamine Yamal,lamine yamal,Barcelona,barcelona
4,Kylian Mbappé,kylian mbappe,Real Madrid,real madrid
5,Sergio Arribas,sergio arribas,Real Betis,real betis
6,Unai Marrero,unai marrero,Real Sociedad,real sociedad
7,Pedri,pedri,Barcelona,barcelona
8,Isco,isco,Real Betis,real betis
9,Alejandro Baena,alejandro baena,Villarreal,villarreal


In [100]:
print("Duplicados en Capology por jugador-equipo:", df_sal.duplicated(subset=["player_norm", "team_norm"]).sum())
print("Duplicados en Sofascore por jugador-equipo:", df_perf.duplicated(subset=["player_norm", "team_norm"]).sum())

Duplicados en Capology por jugador-equipo: 0
Duplicados en Sofascore por jugador-equipo: 0


In [101]:
teams_perf = sorted(df_perf["team_norm"].dropna().unique())
teams_sal = sorted(df_sal["team_norm"].dropna().unique())

print("=== Equipos en Sofascore ===")
print(teams_perf)

print("\n=== Equipos en Capology ===")
print(teams_sal)

=== Equipos en Sofascore ===
['athletic club', 'atletico madrid', 'barcelona', 'celta vigo', 'deportivo alaves', 'espanyol', 'getafe', 'girona fc', 'las palmas', 'leganes', 'mallorca', 'osasuna', 'rayo vallecano', 'real betis', 'real madrid', 'real sociedad', 'real valladolid', 'sevilla', 'valencia', 'villarreal']

=== Equipos en Capology ===
['alaves', 'athletic club', 'atletico madrid', 'barcelona', 'celta vigo', 'espanyol', 'getafe', 'girona', 'las palmas', 'leganes', 'mallorca', 'osasuna', 'rayo vallecano', 'real betis', 'real madrid', 'real sociedad', 'sevilla', 'valencia', 'valladolid', 'villarreal']


In [102]:
df_perf["team_norm_match"] = df_perf["team_norm"]
df_sal["team_norm_match"] = df_sal["team_norm"]

C:\Users\User\AppData\Local\Temp\ipykernel_22864\1590773313.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_perf["team_norm_match"] = df_perf["team_norm"]


In [103]:
team_mapping_perf = {
    "deportivo alaves": "alaves",
    "girona fc": "girona",
    "real valladolid": "valladolid"
}

team_mapping_sal = {}

In [104]:
df_perf["team_norm_match"] = df_perf["team_norm"].replace(team_mapping_perf)
df_sal["team_norm_match"] = df_sal["team_norm"].replace(team_mapping_sal)

In [105]:
print("=== Equipos en Sofascore corregidos ===")
print(sorted(df_perf["team_norm_match"].dropna().unique()))

print("\n=== Equipos en Capology corregidos ===")
print(sorted(df_sal["team_norm_match"].dropna().unique()))

=== Equipos en Sofascore corregidos ===
['alaves', 'athletic club', 'atletico madrid', 'barcelona', 'celta vigo', 'espanyol', 'getafe', 'girona', 'las palmas', 'leganes', 'mallorca', 'osasuna', 'rayo vallecano', 'real betis', 'real madrid', 'real sociedad', 'sevilla', 'valencia', 'valladolid', 'villarreal']

=== Equipos en Capology corregidos ===
['alaves', 'athletic club', 'atletico madrid', 'barcelona', 'celta vigo', 'espanyol', 'getafe', 'girona', 'las palmas', 'leganes', 'mallorca', 'osasuna', 'rayo vallecano', 'real betis', 'real madrid', 'real sociedad', 'sevilla', 'valencia', 'valladolid', 'villarreal']


In [106]:
df_merge = df_perf.merge(
    df_sal,
    left_on=["player_norm", "team_norm_match"],
    right_on=["player_norm", "team_norm_match"],
    how="left",
    suffixes=("_perf", "_sal")
)

print(df_merge.shape)
display(df_merge.head())

(589, 128)


,player_perf,player id,team id,team_perf,data_country,data_season,accurateChippedPasses,accurateCrosses,accurateCrossesPercentage,accurateFinalThirdPasses,...,team_norm_match,player_sal,weekly_salary,yearly_salary,adjusted_gross_salary,position_capology,age,country,team_sal,team_norm_sal
0,Aitor Fernández,99516,2820,Osasuna,spain,2425,3,0,0.00,3,...,osasuna,Aitor Fernández,16154.0,840000.0,"€ 840,000",K,33.0,Spain,Osasuna,osasuna
1,Leo Román,1131909,2826,Mallorca,spain,2425,8,0,0.00,11,...,mallorca,Leo Román,5192.0,270000.0,"€ 270,000",K,24.0,Spain,Mallorca,mallorca
2,Raphinha,831005,2817,Barcelona,spain,2425,62,53,24.65,511,...,barcelona,Raphinha,320577.0,16670000.0,"€ 16,670,000",F,28.0,Brazil,Barcelona,barcelona
3,Lamine Yamal,1402912,2817,Barcelona,spain,2425,45,22,20.37,633,...,barcelona,Lamine Yamal,64038.0,3330000.0,"€ 3,330,000",F,17.0,Spain,Barcelona,barcelona
4,Kylian Mbappé,826643,2829,Real Madrid,spain,2425,49,6,17.14,573,...,real madrid,Kylian Mbappé,600962.0,31250000.0,"€ 31,250,000",F,26.0,France,Real Madrid,real madrid


In [107]:
matched = df_merge["yearly_salary"].notna().sum()
total = len(df_merge)
match_rate = matched / total * 100

print(f"Jugadores en Sofascore: {total}")
print(f"Jugadores con salario asignado: {matched}")
print(f"Tasa de match: {match_rate:.2f}%")

Jugadores en Sofascore: 589
Jugadores con salario asignado: 487
Tasa de match: 82.68%


In [108]:
print(df_merge.columns.tolist())

['player_perf', 'player id', 'team id', 'team_perf', 'data_country', 'data_season', 'accurateChippedPasses', 'accurateCrosses', 'accurateCrossesPercentage', 'accurateFinalThirdPasses', 'accurateLongBalls', 'accurateLongBallsPercentage', 'accurateOppositionHalfPasses', 'accurateOwnHalfPasses', 'accuratePasses', 'accuratePassesPercentage', 'aerialDuelsWon', 'aerialDuelsWonPercentage', 'aerialLost', 'appearances', 'assists', 'attemptPenaltyMiss', 'attemptPenaltyPost', 'attemptPenaltyTarget', 'ballRecovery', 'bigChancesCreated', 'bigChancesMissed', 'blockedShots', 'cleanSheet', 'clearances', 'countRating', 'crossesNotClaimed', 'directRedCards', 'dispossessed', 'dribbledPast', 'duelLost', 'errorLeadToGoal', 'errorLeadToShot', 'expectedAssists', 'expectedGoals', 'fouls', 'freeKickGoal', 'goalConversionPercentage', 'goalKicks', 'goals', 'goalsAssistsSum', 'goalsConceded', 'goalsConcededInsideTheBox', 'goalsConcededOutsideTheBox', 'goalsFromInsideTheBox', 'goalsFromOutsideTheBox', 'goalsPreven

In [109]:
no_match = df_merge[df_merge["yearly_salary"].isna()].copy()

cols_to_show = [col for col in [
    "player_perf", "team_perf", "player_norm", "team_norm_match"
] if col in no_match.columns]

print("Total sin match:", len(no_match))
display(no_match[cols_to_show].head(30))

Total sin match: 102


,player_perf,team_perf,player_norm,team_norm_match
5,Sergio Arribas,Real Betis,sergio arribas,real betis
9,Alejandro Baena,Villarreal,alejandro baena,villarreal
27,Dani Raba,Leganés,dani raba,leganes
37,Yan Diomande,Leganés,yan diomande,leganes
38,Daniel Vivian,Athletic Club,daniel vivian,athletic club
64,Viktor Tsygankov,Girona FC,viktor tsygankov,girona
112,Angel Ortiz,Real Betis,angel ortiz,real betis
113,Iker Almena,Girona FC,iker almena,girona
117,José María Giménez,Atlético Madrid,jose maria gimenez,atletico madrid
118,José Luis Gayà,Valencia,jose luis gaya,valencia


In [110]:
with_match = df_merge[df_merge["yearly_salary"].notna()].copy()

print("Total con match:", len(with_match))
display(with_match.head(20))

Total con match: 487


,player_perf,player id,team id,team_perf,data_country,data_season,accurateChippedPasses,accurateCrosses,accurateCrossesPercentage,accurateFinalThirdPasses,...,team_norm_match,player_sal,weekly_salary,yearly_salary,adjusted_gross_salary,position_capology,age,country,team_sal,team_norm_sal
0,Aitor Fernández,99516,2820,Osasuna,spain,2425,3,0,0.00,3,...,osasuna,Aitor Fernández,16154.0,840000.0,"€ 840,000",K,33.0,Spain,Osasuna,osasuna
1,Leo Román,1131909,2826,Mallorca,spain,2425,8,0,0.00,11,...,mallorca,Leo Román,5192.0,270000.0,"€ 270,000",K,24.0,Spain,Mallorca,mallorca
2,Raphinha,831005,2817,Barcelona,spain,2425,62,53,24.65,511,...,barcelona,Raphinha,320577.0,16670000.0,"€ 16,670,000",F,28.0,Brazil,Barcelona,barcelona
3,Lamine Yamal,1402912,2817,Barcelona,spain,2425,45,22,20.37,633,...,barcelona,Lamine Yamal,64038.0,3330000.0,"€ 3,330,000",F,17.0,Spain,Barcelona,barcelona
4,Kylian Mbappé,826643,2829,Real Madrid,spain,2425,49,6,17.14,573,...,real madrid,Kylian Mbappé,600962.0,31250000.0,"€ 31,250,000",F,26.0,France,Real Madrid,real madrid
6,Unai Marrero,1094782,2824,Real Sociedad,spain,2425,4,0,0.00,0,...,real sociedad,Unai Marrero,3462.0,180000.0,"€ 180,000",K,23.0,Spain,Real Sociedad,real sociedad
7,Pedri,992587,2817,Barcelona,spain,2425,132,10,17.24,918,...,barcelona,Pedri,240385.0,12500000.0,"€ 12,500,000",M,22.0,Spain,Barcelona,barcelona
8,Isco,103417,2816,Real Betis,spain,2425,70,27,29.35,453,...,real betis,Isco,120192.0,6250000.0,"€ 6,250,000",F,32.0,Spain,Real Betis,real betis
10,Federico Valverde,831808,2829,Real Madrid,spain,2425,98,2,7.69,555,...,real madrid,Federico Valverde,320577.0,16670000.0,"€ 16,670,000",M,26.0,Uruguay,Real Madrid,real madrid
11,Jude Bellingham,991011,2829,Real Madrid,spain,2425,45,1,5.56,489,...,real madrid,Jude Bellingham,400577.0,20830000.0,"€ 20,830,000",F,21.0,England,Real Madrid,real madrid
